In [1]:
# ACRS LAYER 3 — Agentic Credit Risk Supervisor
# Three autonomous agents that work together:
#   Agent 1: Scoring Agent  — predict + explain each loan
#   Agent 2: Drift Agent    — detect model/data drift in real time
#   Agent 3: Compliance Agent — HITL trigger + audit trail

import numpy as np
import pandas as pd
import pickle, shap, json, csv, os, warnings
from datetime import datetime
from sklearn.metrics import roc_auc_score, f1_score
from scipy.stats import ks_2samp
from collections import Counter
warnings.filterwarnings('ignore')

X_train = pd.read_parquet('acrs_data/X_train.parquet')
X_val   = pd.read_parquet('acrs_data/X_val.parquet')
X_test  = pd.read_parquet('acrs_data/X_test.parquet')
y_test  = np.load('acrs_data/y_test.npy')

# Load models
with open('acrs_data/xgb_final.pkl',  'rb') as f: xgb_final  = pickle.load(f)
with open('acrs_data/lgbm_final.pkl', 'rb') as f: lgbm_final = pickle.load(f)
with open('acrs_data/rf_final.pkl',   'rb') as f: rf_final   = pickle.load(f)

# Feature verification
assert X_test.shape[1] == 82, f"Expected 82 features, got {X_test.shape[1]}"

# Ensemble helper
def ensemble_predict(X):
    return (xgb_final.predict_proba(X)[:,1] +
            lgbm_final.predict_proba(X)[:,1] +
            rf_final.predict_proba(X)[:,1]) / 3

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print("82 clean features confirmed — ready for Layer 3")

X_train: (449858, 82) | X_test: (504194, 82)
82 clean features confirmed — ready for Layer 3


In [3]:
class ScoringAgent:
    """
    Agent 1: Real-time scoring + SHAP explanation per loan.
    - Computes ensemble probability of default
    - Assigns risk label (Low / Medium / High)
    - Returns top-3 SHAP drivers for the decision
    """
    def __init__(self, xgb_model, lgbm_model, rf_model):
        self.xgb  = xgb_model
        self.lgbm = lgbm_model
        self.rf   = rf_model
        self.explainer = shap.TreeExplainer(xgb_model)

    def score(self, loan_row):
        prob  = (self.xgb.predict_proba(loan_row)[:,1] +
                 self.lgbm.predict_proba(loan_row)[:,1] +
                 self.rf.predict_proba(loan_row)[:,1]) / 3
        score = float(prob[0])

        if score < 0.30:   label = "Low Risk"
        elif score < 0.50: label = "Medium Risk"
        else:              label = "High Risk"

        shap_vals = self.explainer.shap_values(loan_row)[0]
        shap_df   = pd.DataFrame({'feature': loan_row.columns, 'shap': shap_vals})
        top3      = shap_df.reindex(shap_df['shap'].abs().nlargest(3).index)

        return {
            'score':     round(score, 4),
            'label':     label,
            'top3_shap': top3[['feature', 'shap']].values.tolist()
        }

agent1 = ScoringAgent(xgb_final, lgbm_final, rf_final)
print("Agent 1 (Scoring) initialised")

# Demo: 3 loans
print("\n--- Agent 1 Demo: Scoring 3 loans ---")
for i in [0, 1, 2]:
    loan   = X_test.iloc[[i]]
    result = agent1.score(loan)
    print(f"\nLoan #{i}  |  True label: {'DEFAULT' if y_test[i]==1 else 'PAID'}")
    print(f"  Score : {result['score']}  →  {result['label']}")
    for feat, val in result['top3_shap']:
        print(f"    {feat:<35} SHAP={val:+.4f}  {'↑ risk' if val>0 else '↓ risk'}")

Agent 1 (Scoring) initialised

--- Agent 1 Demo: Scoring 3 loans ---

Loan #0  |  True label: PAID
  Score : 0.2149  →  Low Risk
    sub_grade                           SHAP=+0.3561  ↑ risk
    term                                SHAP=-0.2139  ↓ risk
    annual_inc                          SHAP=-0.1753  ↓ risk

Loan #1  |  True label: PAID
  Score : 0.3494  →  Medium Risk
    term                                SHAP=+0.3297  ↑ risk
    dti                                 SHAP=+0.2818  ↑ risk
    sub_grade                           SHAP=+0.2027  ↑ risk

Loan #2  |  True label: PAID
  Score : 0.0321  →  Low Risk
    sub_grade                           SHAP=-0.6708  ↓ risk
    annual_inc                          SHAP=-0.2548  ↓ risk
    int_rate                            SHAP=-0.2493  ↓ risk


In [5]:
class DriftAgent:
    """
    Agent 2: Online drift detection using 3 sequential detectors.
    - ADWIN:        detects mean score shift over sliding window
    - Page-Hinkley: detects persistent upward shift in scores
    - DDM:          detects rising error rate (with cooldown to prevent alert fatigue)
    """
    def __init__(self, window_size=500, adwin_threshold=0.03,
                 ph_threshold=50, ddm_drift=3.0, ddm_cooldown=500):
        self.adwin_window      = []
        self.adwin_window_size = window_size
        self.adwin_ref_mean    = None
        self.adwin_threshold   = adwin_threshold

        self.ph_sum      = 0.0
        self.ph_min      = 0.0
        self.ph_threshold= ph_threshold
        self.ph_mean_ref = None

        self.ddm_n          = 0
        self.ddm_errors     = 0
        self.ddm_min_rate   = float('inf')
        self.ddm_min_std    = float('inf')
        self.ddm_drift      = ddm_drift
        self.ddm_cooldown   = ddm_cooldown
        self.ddm_last_alert = -ddm_cooldown

        self.alerts = []

    def update(self, score, true_label, idx):
        status = "OK"

        # ADWIN
        self.adwin_window.append(score)
        if len(self.adwin_window) > self.adwin_window_size:
            self.adwin_window.pop(0)
        if len(self.adwin_window) == self.adwin_window_size:
            win_mean = np.mean(self.adwin_window)
            if self.adwin_ref_mean is None:
                self.adwin_ref_mean = win_mean
            elif abs(win_mean - self.adwin_ref_mean) > self.adwin_threshold:
                status = "ADWIN_DRIFT"
                self.adwin_ref_mean = win_mean

        # Page-Hinkley
        if self.ph_mean_ref is None:
            self.ph_mean_ref = score
        self.ph_sum += score - self.ph_mean_ref - 0.01
        self.ph_min  = min(self.ph_min, self.ph_sum)
        if (self.ph_sum - self.ph_min) > self.ph_threshold:
            status = "PH_DRIFT"
            self.ph_sum = 0.0; self.ph_min = 0.0
            self.ph_mean_ref = score

        # DDM with cooldown
        self.ddm_n      += 1
        self.ddm_errors += int((1 if score >= 0.5 else 0) != true_label)
        if self.ddm_n > 200:
            rate = self.ddm_errors / self.ddm_n
            std  = np.sqrt(rate * (1 - rate) / self.ddm_n)
            if rate + std <= self.ddm_min_rate + self.ddm_min_std:
                self.ddm_min_rate = rate; self.ddm_min_std = std
            elif (rate + std > self.ddm_min_rate + self.ddm_drift * self.ddm_min_std
                  and idx - self.ddm_last_alert >= self.ddm_cooldown):
                status = "DDM_DRIFT"
                self.ddm_last_alert = idx

        if status != "OK":
            self.alerts.append({'idx': idx, 'type': status, 'score': round(score, 4)})
        return status

# Run Agent 2 on 5000 loans
agent2 = DriftAgent()
scores_5k = ensemble_predict(X_test.iloc[:5000])
for i, (score, label) in enumerate(zip(scores_5k, y_test[:5000])):
    agent2.update(score, label, i)

alert_counts = Counter(a['type'] for a in agent2.alerts)
print(f"Drift alerts in first 5,000 loans: {len(agent2.alerts)}")
for t, c in alert_counts.items():
    print(f"  {t}: {c}")
print("\nAlert log:")
for a in agent2.alerts:
    print(f"  Loan #{a['idx']:>5} | {a['type']:<15} | score={a['score']}")

Drift alerts in first 5,000 loans: 10
  DDM_DRIFT: 9
  ADWIN_DRIFT: 1

Alert log:
  Loan #  526 | DDM_DRIFT       | score=0.2644
  Loan # 1026 | DDM_DRIFT       | score=0.2666
  Loan # 1526 | DDM_DRIFT       | score=0.1025
  Loan # 2026 | DDM_DRIFT       | score=0.2175
  Loan # 2526 | DDM_DRIFT       | score=0.061
  Loan # 2911 | ADWIN_DRIFT     | score=0.4085
  Loan # 3026 | DDM_DRIFT       | score=0.3446
  Loan # 3526 | DDM_DRIFT       | score=0.0295
  Loan # 4026 | DDM_DRIFT       | score=0.0747
  Loan # 4526 | DDM_DRIFT       | score=0.0485


In [7]:
class ComplianceAgent:
    """
    Agent 3: Governance layer.
    - Triggers HITL review for borderline, drifting, or extreme-risk loans
    - Writes every decision to an immutable audit trail (CSV)
    - Aligned with SR 26-2 (Federal Reserve, April 2026),
      GDPR Art.22, and EBA model risk guidelines
    """
    AUDIT_FILE = 'acrs_data/audit_trail.csv'
    HEADER = ['timestamp', 'loan_idx', 'score', 'risk_label',
              'drift_status', 'hitl_required', 'hitl_reason',
              'top1_feature', 'top1_shap', 'decision']

    def __init__(self, grey_low=0.40, grey_high=0.60, high_risk=0.75):
        self.grey_low  = grey_low
        self.grey_high = grey_high
        self.high_risk = high_risk
        self.hitl_count  = 0
        self.total_count = 0

        # Fresh audit trail on each run
        with open(self.AUDIT_FILE, 'w', newline='') as f:
            csv.writer(f).writerow(self.HEADER)

    def evaluate(self, loan_idx, agent1_result, drift_status):
        score  = agent1_result['score']
        label  = agent1_result['label']
        top1   = agent1_result['top3_shap'][0]

        hitl, reason = False, "None"
        if self.grey_low <= score <= self.grey_high:
            hitl, reason = True, "Grey zone score (borderline)"
        elif drift_status != "OK":
            hitl, reason = True, f"Drift detected: {drift_status}"
        elif score >= self.high_risk:
            hitl, reason = True, "Extreme high-risk score"

        decision = ("REFER_TO_HUMAN" if hitl
                    else "AUTO_APPROVE" if score < 0.30
                    else "AUTO_DECLINE")

        self.total_count += 1
        if hitl: self.hitl_count += 1

        with open(self.AUDIT_FILE, 'a', newline='') as f:
            csv.writer(f).writerow([
                datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                loan_idx, score, label, drift_status,
                hitl, reason, top1[0], round(top1[1], 4), decision
            ])
        return {'decision': decision, 'hitl': hitl, 'hitl_reason': reason}

agent3 = ComplianceAgent()
print("Agent 3 (Compliance) initialised — fresh audit trail created")

# Full pipeline demo: 500 loans
print("\n--- ACRS Full Pipeline Demo (2000 loans) ---")
agent2_demo = DriftAgent()
summary = {'AUTO_APPROVE': 0, 'AUTO_DECLINE': 0, 'REFER_TO_HUMAN': 0}

for i in range(2000):
    loan       = X_test.iloc[[i]]
    a1_result  = agent1.score(loan)
    drift_stat = agent2_demo.update(a1_result['score'], y_test[i], i)
    a3_result  = agent3.evaluate(i, a1_result, drift_stat)
    summary[a3_result['decision']] += 1

print(f"\nDecision summary (2000 loans):")
for k, v in summary.items():
    pct = v / agent3.total_count * 100
    print(f"  {k:<20}: {v:>4}  ({pct:.1f}%)")

hitl_rate = agent3.hitl_count / agent3.total_count * 100
print(f"\nHITL rate: {agent3.hitl_count}/{agent3.total_count} = {hitl_rate:.1f}%")

audit_df = pd.read_csv(agent3.AUDIT_FILE)
print(f"Audit trail: {len(audit_df)} rows logged")
print(audit_df.tail(3).to_string(index=False))

Agent 3 (Compliance) initialised — fresh audit trail created

--- ACRS Full Pipeline Demo (2000 loans) ---

Decision summary (2000 loans):
  AUTO_APPROVE        : 1805  (90.2%)
  AUTO_DECLINE        :  133  (6.7%)
  REFER_TO_HUMAN      :   62  (3.1%)

HITL rate: 62/2000 = 3.1%
Audit trail: 2000 rows logged
          timestamp  loan_idx  score risk_label drift_status  hitl_required hitl_reason top1_feature  top1_shap     decision
2026-11-05 18:23:44      1997 0.2144   Low Risk           OK          False         NaN         term     0.3438 AUTO_APPROVE
2026-11-05 18:23:44      1998 0.1460   Low Risk           OK          False         NaN    sub_grade    -0.3557 AUTO_APPROVE
2026-11-05 18:23:44      1999 0.0405   Low Risk           OK          False         NaN    sub_grade    -0.7092 AUTO_APPROVE


In [9]:
print("Running full test set through ACRS pipeline...")
all_scores = ensemble_predict(X_test)

auc_l3  = roc_auc_score(y_test, all_scores)
ks_l3   = ks_2samp(all_scores[y_test==1], all_scores[y_test==0]).statistic
gini_l3 = 2 * auc_l3 - 1
f1_l3   = f1_score(y_test, (all_scores >= 0.5).astype(int))

# Dynamic drift summary
drift_total  = len(agent2.alerts)
drift_counts = Counter(a['type'] for a in agent2.alerts)
drift_summary = ", ".join(f"{c} {t}" for t, c in drift_counts.items())

print("\n" + "=" * 60)
print("LAYER 3 — ACRS RESULTS SUMMARY")
print("=" * 60)
print(f"  Predictive Performance (Test Set 2016+):")
print(f"    AUC-ROC  : {auc_l3:.4f}")
print(f"    KS Stat  : {ks_l3:.4f}")
print(f"    Gini     : {gini_l3:.4f}")
print(f"    F1 Score : {f1_l3:.4f}")
print(f"\n  Agentic Supervisory Capability:")
print(f"    Agent 1 — Real-time SHAP per loan (top-3 drivers)")
print(f"    Agent 2 — {drift_total} drift alerts in 5,000 loans ({drift_summary})")
print(f"    Agent 3 — HITL rate: {hitl_rate:.1f}% | {len(audit_df)} audit rows")
print(f"\n  Regulatory Alignment:")
print(f"    SR 26-2 (Federal Reserve, April 2026)")
print(f"    GDPR Art.22 | EBA Model Risk Guidelines")
print("=" * 60)

l3_results = {
    "layer": 3,
    "auc":  round(auc_l3, 4),
    "ks":   round(ks_l3, 4),
    "gini": round(gini_l3, 4),
    "f1":   round(f1_l3, 4),
    "drift_alerts_5k": drift_total,
    "drift_breakdown": dict(drift_counts),
    "hitl_rate_pct":   round(hitl_rate, 1),
    "audit_rows":      len(audit_df)
}
with open('acrs_data/layer3_results.json', 'w') as f:
    json.dump(l3_results, f, indent=2)
print("Saved: layer3_results.json")
print("Layer 3 COMPLETE")

Running full test set through ACRS pipeline...

LAYER 3 — ACRS RESULTS SUMMARY
  Predictive Performance (Test Set 2016+):
    AUC-ROC  : 0.7057
    KS Stat  : 0.2974
    Gini     : 0.4115
    F1 Score : 0.0345

  Agentic Supervisory Capability:
    Agent 1 — Real-time SHAP per loan (top-3 drivers)
    Agent 2 — 10 drift alerts in 5,000 loans (9 DDM_DRIFT, 1 ADWIN_DRIFT)
    Agent 3 — HITL rate: 3.1% | 2000 audit rows

  Regulatory Alignment:
    SR 26-2 (Federal Reserve, April 2026)
    GDPR Art.22 | EBA Model Risk Guidelines
Saved: layer3_results.json
Layer 3 COMPLETE
